In [ ]:
from modelscope import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftConfig, PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B"
peft_conf_id  = "./trading_intent_classification_lora/lora_adapter"

# 加载PEFT模型
base_model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=base_model_id, num_labels=2).to("cuda")
tokenizer  = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=base_model_id)

peft_conf  = PeftConfig.from_pretrained(pretrained_model_name_or_path=peft_conf_id)
peft_model = PeftModel.from_pretrained(base_model, peft_conf_id)
peft_model

2026-03-16 16:33:30,681 - modelscope - INFO - Target directory already exists, skipping creation.
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at /root/.cache/modelscope/hub/models/Qwen/Qwen2___5-0___5B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-03-16 16:33:32,070 - modelscope - INFO - Target directory already exists, skipping creation.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Qwen2ForSequenceClassification(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.08, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [ ]:
import pandas as pd
import torch
from sklearn.metrics import accuracy_score

device = "cpu"

tests_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/datasets.csv")
tests_df

all_samples = len(tests_df)
correct_num = 0
peft_model.eval()

for i in range(0, all_samples):
    token_ids = tokenizer([tests_df['Text'][i]], return_tensors="pt", padding=True, truncation=True).to(device)
    if accuracy_score(y_pred=torch.argmax(input=peft_model(**token_ids).logits, dim=-1).to(device).numpy(), y_true=[tests_df['Label'][i]]) == 1.0:
        correct_num+=1
accuracy = f"All Accuracy:{correct_num / all_samples}"
accuracy

KeyboardInterrupt: 